# Parse an ASPECT model

`AspectModelParser` collects timestep records, timing tables, and parameter dictionaries from a model output directory. This notebook creates a temporary example, so it can run without an existing simulation.

For your own model, use `AspectModelParser("/path/to/model", output_directory="output")` and call `parse()`. The output directory defaults to `output`; a relative path is resolved against the model directory, and an absolute path is also accepted.

The parser also looks for an optional World Builder file, `original.wb`, in the output directory. When present, it loads the file into `world_builder_parameters`; otherwise, this attribute is `None`.


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt

from gdmate.aspect_tools import AspectModelParser


## Create example output files

The required files are `log.txt`, `original.prm`, and `parameters.json`. The parser reads `original.prm` with gdmate's parameter parser. The full parameters in `parameters.json` are loaded separately, preserving the JSON structure and values. JSON files must contain objects.


In [ ]:
temporary_directory = TemporaryDirectory()
model_directory = Path(temporary_directory.name)
output = model_directory / "output"
output.mkdir()

(output / "original.prm").write_text(
    "set Dimension = 2\n"
    "subsection Geometry model\n"
    "  set Model name = box\n"
    "end\n",
    encoding="utf-8",
)
(output / "parameters.json").write_text(
    json.dumps({"Dimension": "2", "Output directory": "output", "End time": "250"}),
    encoding="utf-8",
)
(output / "original.wb").write_text(
    json.dumps({"version": "1.0", "features": []}), encoding="utf-8"
)
(output / "log.txt").write_text(
    """--     . version 3.1.0-pre
*** Timestep 0: t=0 years, dt=0 years
*** Timestep 1: t=100 years, dt=100 years
*** Timestep 2: t=250 years, dt=150 years
| Total wallclock time elapsed since start | 10s | |
| Section | no. calls | wall time | % of total |
+---------+-----------+-----------+------------+
| Solve Stokes system | 2 | 5s | 50% |
+---------+-----------+-----------+------------+
""",
    encoding="utf-8",
)


## Load the model

Construction sets paths and initializes empty collections. `parse()` reads the files and replaces the stored data. Calling it again refreshes the collections rather than appending records. Version validation is inherited from `AspectLogParser`: ASPECT major version 3 is required by default. Use `validate_version=False` only when deliberately reading older logs or fragments.


In [ ]:
model = AspectModelParser(model_directory)
model.parse()

print("Output directory:", model.output_directory)
print("Original parameters:", model.original_parameters)
print("Full parameters:", model.parameters)
print("World Builder settings:", model.world_builder_parameters)

assert model.original_parameters["Geometry model"]["Model name"] == "box"
assert model.parameters["End time"] == "250"
assert model.world_builder_parameters["features"] == []


## Inspect timesteps and timing blocks

Each timestep records its number, model time, timestep size, time unit, and log line number. Records remain in file order; repeated timestep numbers after restarts are preserved. No time conversion is performed. Timing blocks retain their total wall time, individual sections, and associated timestep number. Logs with no timing tables produce an empty list.


In [ ]:
for step in model.timesteps:
    print(f"Step {step.number}: t={step.time} {step.time_unit}, "
          f"dt={step.timestep_size} {step.time_unit}")

for block in model.timing_blocks:
    print(f"Timing at step {block.timestep_number}: "
          f"{block.total_wallclock_seconds} seconds")
    for section in block.sections:
        print(section.name, section.wall_time_seconds)

assert [step.time for step in model.timesteps] == [0, 100, 250]
assert model.timing_blocks[0].timestep_number == 2


In [ ]:
units = {step.time_unit for step in model.timesteps}
assert len(units) == 1, "Convert mixed time units before plotting."
fig, ax = plt.subplots()
ax.plot([step.number for step in model.timesteps],
        [step.time for step in model.timesteps], marker="o")
ax.set_xlabel("Timestep number")
ax.set_ylabel(f"Model time ({next(iter(units))})")
ax.grid(True)
plt.show()
plt.close(fig)

temporary_directory.cleanup()
